# Model iterations and calibration

Privacy rule: no subject, body, preview, or attachment content is loaded.

In [ ]:
PROJECT_ROOT = '..'
RANDOM_SEED = 365

In [ ]:
from pathlib import Path
ROOT = Path(PROJECT_ROOT).resolve()
DATA_ROOT = ROOT / 'data'
ARTIFACT_ROOT = ROOT / 'artifacts/models/current'

In [ ]:
from packages.ml.m365risk_ml.train import train
manifest = train(DATA_ROOT, ARTIFACT_ROOT)
manifest['status'], manifest['selected_model']

In [ ]:
import json, pandas as pd, matplotlib.pyplot as plt
metrics = json.loads((ARTIFACT_ROOT/'metrics.json').read_text())
models = pd.DataFrame(metrics['email_models']); display(models)
models.set_index('name')[['pr_auc','roc_auc','brier','ece']].plot.bar(subplots=True, figsize=(10,9), title='Iteration metrics'); plt.tight_layout()

In [ ]:
from sklearn.calibration import calibration_curve
from sklearn.metrics import ConfusionMatrixDisplay, precision_recall_curve
pred = pd.read_parquet(ARTIFACT_ROOT/'evaluation_predictions.parquet'); fig, axes = plt.subplots(1,3,figsize=(16,4))
for name in ['rules','logistic','hist_gradient_boosting']:
    observed, forecast = calibration_curve(pred.label, pred[name], n_bins=10); axes[0].plot(forecast, observed, marker='o', label=name)
axes[0].plot([0,1],[0,1],'--',color='gray'); axes[0].set_title('Calibration'); axes[0].legend()
selected = manifest['selected_model']; threshold = metrics['selected']['threshold']; ConfusionMatrixDisplay.from_predictions(pred.label, pred[selected]>=threshold, ax=axes[1], colorbar=False)
p, r, t = precision_recall_curve(pred.label, pred[selected]); axes[2].plot(t,p[:-1],label='precision'); axes[2].plot(t,r[:-1],label='recall'); axes[2].axvline(threshold,color='red',ls='--'); axes[2].set_title('Threshold sensitivity'); axes[2].legend(); plt.tight_layout()

In [ ]:
from IPython.display import Image, display
display(Image(filename=ARTIFACT_ROOT/'model_comparison.png'))